# RV3 joint damage SE-ResNeXt50 (EMA+SWA+sampler, 608px, adaptive cosine)

In [ ]:
import subprocess, sys, os, json, time, traceback
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = Path("/tmp/RescueVision")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/AliOmarAbdelhady/RescueVision.git", str(REPO)], check=True)
sys.path.insert(0, str(REPO / "src"))

def mark(stage, ok=True, extra=""):
    """Progress ledger in /kaggle/working: survives kills where the log does not."""
    rec = {"t": time.strftime("%H:%M:%S"), "stage": stage, "ok": bool(ok), "extra": str(extra)[:1500]}
    with open(WORK / "_progress.txt", "a") as f:
        f.write(json.dumps(rec) + "\n")
    (WORK / "_status.json").write_text(json.dumps(rec, indent=1))
    print("[mark]", rec, flush=True)

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
mark("setup:start")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "albumentations==2.0.8"], check=False)
import albumentations as _A
mark("setup:albumentations", extra=_A.__version__)
print("albumentations", _A.__version__)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch"], check=False)
import segmentation_models_pytorch as smp
print("smp", smp.__version__)
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NW = 0
PIN = False
print("torch", torch.__version__, "| device:", DEVICE)
print(subprocess.run(["df", "-h", "/tmp", "/kaggle/working"], capture_output=True, text=True).stdout)
mark("setup:done", extra=str(DEVICE))

import rescuevision.training.logging_utils as _lu
_orig_log = _lu.CSVLogger.log
def _marked_log(self, metrics, epoch):
    try:
        mark(f"epoch{epoch}", extra=str({k: round(float(v), 4) for k, v in metrics.items()
                                         if isinstance(v, (int, float))})[:400])
    except Exception:
        pass
    return _orig_log(self, metrics, epoch)
_lu.CSVLogger.log = _marked_log
print("epoch heartbeat patch applied")

In [ ]:
import numpy as np
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2

from rescuevision.constants import IMAGENET_MEAN, IMAGENET_STD
from rescuevision.data import datasets as _D
from PIL import Image as _PIL
from pathlib import Path as _PPath

def _load_arr(path):
    path = _PPath(path)
    if path.suffix == ".npy":
        return np.load(path)
    return np.array(_PIL.open(path))

def _find_tile(directory, tile_id):
    for ext in (".png", ".jpg", ".jpeg", ".npy"):
        p = _PPath(directory) / f"{tile_id}{ext}"
        if p.exists():
            return p
    raise FileNotFoundError(f"{tile_id} in {directory}")

def _tile_dataset_init(self, processed_dir, split_csv, split="train", mask_type="building_binary",
                       transform=None, image_size=512, debug_limit=None):
    import pandas as pd
    self.processed_dir = _PPath(processed_dir)
    self.images_dir = self.processed_dir / "images" / "pre"
    self.pre_dir = self.processed_dir / "images" / "pre"
    self.post_dir = self.processed_dir / "images" / "post"
    if type(self).__name__ == "XBDDamageSegmentationDataset":
        self.masks_dir = self.processed_dir / "masks" / "damage_multiclass"
    elif type(self).__name__ == "XBDChangeDetectionDataset":
        self.masks_dir = self.processed_dir / "masks" / "change_binary"
    else:
        self.masks_dir = self.processed_dir / "masks" / mask_type
    self.transform = transform
    self.image_size = image_size
    splits_df = pd.read_csv(split_csv)
    tile_ids = splits_df[splits_df["split"] == split]["tile_id"].tolist()
    if debug_limit:
        tile_ids = tile_ids[:debug_limit]
    self.tile_ids = [t for t in tile_ids
                     if _find_tile(self.pre_dir, t).exists()
                     and _find_tile(self.post_dir, t).exists()
                     and _find_tile(self.masks_dir, t).exists()]
    print(f"{type(self).__name__}: split={split}, {len(self.tile_ids)} tiles")

def _xbd_seg_getitem(self, idx):
    tile_id = self.tile_ids[idx]
    image = _load_arr(_find_tile(self.pre_dir, tile_id))
    mask = _load_arr(_find_tile(self.masks_dir, tile_id))
    if self.transform:
        aug = self.transform(image=image, mask=mask)
        image, mask = aug["image"], aug["mask"]
    mask = mask.long() if torch.is_tensor(mask) else torch.from_numpy(mask).long()
    return {"image": image, "mask": mask, "tile_id": tile_id}

_D.XBDSegmentationDataset.__init__ = _tile_dataset_init
_D.XBDSegmentationDataset.__getitem__ = _xbd_seg_getitem
mark("patch:done")
print("png-aware dataset patch applied")

In [ ]:
# ---- official xView2 pixel scorer (inlined verbatim from repo = single source) ----
"""Official xView2 pixel-based scoring.

Port of the metric used by the xView2 challenge (DIUx-xView/xView2_scoring):
identical semantics, array-based API so it can score in-memory predictions
without writing the official PNG directory layout.

Scoring rules (match the official implementation exactly):
- Localization F1: pixel counts of binarized building masks
  (pred building > 0 vs target building > 0), summed over ALL tiles globally.
- Damage F1: per tile, predicted damage is zeroed outside PREDICTED buildings
  (gating), then only pixels inside TARGET buildings (damage target > 0) are
  scored; per class 1-4 TP/FN/FP are summed globally and converted to F1.
- damage_f1 = harmonic mean of the four class F1s (epsilon 1e-6).
- score = 0.3 * localization_f1 + 0.7 * damage_f1.
"""

from __future__ import annotations

import numpy as np

__all__ = ["XView2PixelScorer", "f1_from_counts", "harmonic_mean_f1"]

_EPS = 1e-6


def f1_from_counts(tp: int, fp: int, fn: int) -> float:
    """F1 from global TP/FP/FN counts (0 when TP, precision or recall is 0)."""
    if tp <= 0:
        return 0.0
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    if precision == 0.0 or recall == 0.0:
        return 0.0
    return float(2 * precision * recall / (precision + recall))


def harmonic_mean_f1(f1s) -> float:
    """Official damage F1: harmonic mean of the four class F1s (eps 1e-6)."""
    xs = list(f1s)
    return float(len(xs) / sum((x + _EPS) ** -1 for x in xs))


class XView2PixelScorer:
    """Accumulates official xView2 pixel counts over tiles.

    Usage::

        scorer = XView2PixelScorer()
        for tile in tiles:
            scorer.add(loc_pred, dmg_pred, loc_target, dmg_target)
        summary = scorer.summary()

    All inputs are uint8 arrays of identical shape. loc_* are binary building
    masks (0/1, anything >0 counts as building); dmg_* carry grades 0-4 where
    0 = background and 1-4 = no/minor/major/destroyed damage inside buildings.
    """

    def __init__(self):
        self.loc_tp = 0
        self.loc_fp = 0
        self.loc_fn = 0
        self.dmg_tp = np.zeros(5, dtype=np.int64)   # index 1-4 used
        self.dmg_fp = np.zeros(5, dtype=np.int64)
        self.dmg_fn = np.zeros(5, dtype=np.int64)
        self.n_tiles = 0

    def add(self, loc_pred, dmg_pred, loc_target, dmg_target) -> None:
        """Add one tile (any spatial shape, pred and target must match)."""
        loc_pred = np.asarray(loc_pred)
        dmg_pred = np.asarray(dmg_pred)
        loc_target = np.asarray(loc_target)
        dmg_target = np.asarray(dmg_target)
        if not (loc_pred.shape == dmg_pred.shape == loc_target.shape == dmg_target.shape):
            raise ValueError(
                f"shape mismatch: pred {loc_pred.shape}/{dmg_pred.shape} vs "
                f"target {loc_target.shape}/{dmg_target.shape}"
            )
        self.n_tiles += 1

        lp_b = loc_pred > 0
        lt_b = loc_target > 0
        self.loc_tp += int(np.logical_and(lp_b, lt_b).sum())
        self.loc_fp += int(np.logical_and(lp_b, ~lt_b).sum())
        self.loc_fn += int(np.logical_and(~lp_b, lt_b).sum())

        dp = np.where(lp_b, dmg_pred, 0)          # gate damage by predicted buildings
        dt = dmg_target
        inside = dt > 0                            # score only target-building pixels
        dp_in = dp[inside]
        dt_in = dt[inside]
        for c in range(1, 5):
            p_c = dp_in == c
            g_c = dt_in == c
            self.dmg_tp[c] += int(np.logical_and(p_c, g_c).sum())
            self.dmg_fn[c] += int(np.logical_and(~p_c, g_c).sum())
            self.dmg_fp[c] += int(np.logical_and(p_c, ~g_c).sum())

    def summary(self) -> dict:
        localization_f1 = f1_from_counts(self.loc_tp, self.loc_fp, self.loc_fn)
        per_class = {
            c: f1_from_counts(int(self.dmg_tp[c]), int(self.dmg_fp[c]), int(self.dmg_fn[c]))
            for c in range(1, 5)
        }
        damage_f1 = harmonic_mean_f1(per_class.values())
        return {
            "score": 0.3 * localization_f1 + 0.7 * damage_f1,
            "localization_f1": localization_f1,
            "damage_f1": damage_f1,
            "damage_f1_no_damage": per_class[1],
            "damage_f1_minor_damage": per_class[2],
            "damage_f1_major_damage": per_class[3],
            "damage_f1_destroyed": per_class[4],
            "n_tiles": self.n_tiles,
        }

    def __repr__(self) -> str:
        s = self.summary()
        per_class = [s["damage_f1_no_damage"], s["damage_f1_minor_damage"],
                     s["damage_f1_major_damage"], s["damage_f1_destroyed"]]
        return (
            f"XView2PixelScorer(n={s['n_tiles']}) score={s['score']:.4f} "
            f"loc_f1={s['localization_f1']:.4f} dmg_f1={s['damage_f1']:.4f} "
            f"per-class={per_class}"
        )


print("official scorer ready:", XView2PixelScorer)
mark("scorer:done")

In [ ]:
import time
import pandas as pd
mark("tar:wait")
TAR = None
if (Path("/tmp/processed_xbd") / "metadata" / "splits_event_aware.csv").exists():
    print("processed tree already at /tmp/processed_xbd (smoke/debug path)")
else:
    for _attempt in range(40):
        for _p in Path("/kaggle/input").rglob("processed_xbd*.tar"):
            TAR = _p
            break
        if TAR is not None:
            break
        _ents = [p.name for p in Path("/kaggle/input").iterdir()] if Path("/kaggle/input").exists() else []
        print(f"[{_attempt+1}/40] waiting for tar mount, /kaggle/input: {_ents}", flush=True)
        time.sleep(30)
    assert TAR is not None, "processed_xbd_native.tar never mounted - kernel-source attach failed"
    _gib = TAR.stat().st_size / 1e9
    print("tar:", TAR, f"{_gib:.2f} GB")
    mark("tar:mounted", extra=f"{_gib:.2f}GB")
    subprocess.run(["tar", "-xf", str(TAR), "-C", "/tmp"], check=True)
PROCESSED = Path("/tmp/processed_xbd")
assert (PROCESSED / "metadata" / "splits_event_aware.csv").exists(), "extracted tree missing splits csv"
_splits = pd.read_csv(PROCESSED / "metadata" / "splits_event_aware.csv")
print("tiles per split:", _splits["split"].value_counts().to_dict())
_tiles_meta = pd.read_csv(PROCESSED / "metadata" / "tiles.csv")
print("shift columns:", [c for c in _tiles_meta.columns if "shift" in c])
mark("tar:extracted", extra=str(_splits["split"].value_counts().to_dict()))

In [ ]:
# ---- joint damage dataset (6ch siamese input, combined building+grade mask) ----
import pandas as pd
import torch
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2
from rescuevision.constants import IMAGENET_MEAN, IMAGENET_STD


class JointDamageDataset(Dataset):
    """pre+post -> 6ch image; ONE combined mask carries building + grade.
    values: 0 bg | 1 no-dmg | 2 minor | 3 major | 4 destroyed | 6 building w/o grade"""

    def __init__(self, root, split_csv, split="train", transform=None):
        self.root = Path(root)
        df = pd.read_csv(split_csv)
        ids = df[df["split"] == split]["tile_id"].tolist()

        def _ok(t):
            try:
                _find_tile(self.root / "images/pre", t)
                _find_tile(self.root / "masks/damage_multiclass_dil", t)
                return True
            except FileNotFoundError:
                return False

        self.tile_ids = [t for t in ids if _ok(t)]
        self.transform = transform
        print(f"JointDamageDataset split={split}: {len(self.tile_ids)} tiles")

    def __len__(self):
        return len(self.tile_ids)

    def __getitem__(self, idx):
        tid = self.tile_ids[idx]
        pre = _load_arr(_find_tile(self.root / "images/pre", tid))
        post = _load_arr(_find_tile(self.root / "images/post", tid))
        dmg = _load_arr(_find_tile(self.root / "masks/damage_multiclass_dil", tid)).astype(np.uint8)
        loc = _load_arr(self.root / "masks/building_binary" / f"{tid}.png").astype(np.uint8)
        comb = np.where((loc > 0) & (dmg == 0), 6, dmg).astype(np.uint8)
        pair = np.concatenate([pre, post], axis=2)
        if self.transform:
            aug = self.transform(image=pair, mask=comb)
            pair_t, comb_t = aug["image"], aug["mask"]
        else:
            pair_t = torch.from_numpy(pair.transpose(2, 0, 1))
            comb_t = torch.from_numpy(comb)
        comb_t = comb_t.long() if torch.is_tensor(comb_t) else torch.from_numpy(np.asarray(comb_t)).long()
        loc_t = (comb_t > 0).float()
        dmg_t = comb_t.clone() - 1
        dmg_t[(comb_t < 1) | (comb_t > 4)] = 255
        return {"pre": pair_t[:3].float(), "post": pair_t[3:].float(),
                "loc": loc_t, "dmg": dmg_t, "tile_id": tid}


def joint_pair_transforms(phase="train", size=608):
    mean, std = IMAGENET_MEAN * 2, IMAGENET_STD * 2
    tf = [A.Resize(size, size)]
    if phase == "train":
        tf += [A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
               A.Affine(scale=(0.85, 1.15), p=0.4),
               A.RandomBrightnessContrast(p=0.3),
               A.UnsharpMask(p=0.3)]
    tf += [A.Normalize(mean=mean, std=std), ToTensorV2()]
    return A.Compose(tf, is_check_shapes=False)


print("dataset kit ready")
mark("dataset:done")

In [ ]:
# ---- model kit: build SiameseUNet + TTA inference + prob caching ----
import time as _time
import numpy as np
import torch
from torch.utils.data import DataLoader

SIZE = 608

def build_model(encoder="resnet34", weights="none", se_gates=False):
    weights = None if weights in (None, "none", "") else weights
    cls = SiameseUNetSE if se_gates else SiameseUNet
    m = cls(encoder_name=encoder, encoder_weights=weights,
            fusion_mode="concat_absdiff", classes=5)
    return m.to(DEVICE).eval()


@torch.no_grad()
def tta_probs(model, pre, post):
    """4-view TTA -> (loc_prob, dmg_prob[4]) from averaged logits."""
    outs = []
    for k in range(4):
        a, b = pre, post
        if k == 1: a, b = torch.flip(pre, [3]), torch.flip(post, [3])
        elif k == 2: a, b = torch.flip(pre, [2]), torch.flip(post, [2])
        elif k == 3: a, b = torch.flip(pre, [2, 3]), torch.flip(post, [2, 3])
        with torch.amp.autocast(DEVICE.type, enabled=(DEVICE.type == "cuda")):
            o = model(a, b)
        if k == 1: o = torch.flip(o, [3])
        elif k == 2: o = torch.flip(o, [2])
        elif k == 3: o = torch.flip(o, [2, 3])
        outs.append(o.float())
    o = torch.stack(outs, 0).mean(0)
    return torch.sigmoid(o[:, 0]), torch.softmax(o[:, 1:], 1)


@torch.no_grad()
def cache_probs(model, ds, split, limit=None, tag="probs"):
    """TTA inference over ds.tile_ids, per-tile probs cached as float16 npz in /tmp."""
    if limit:
        ds.tile_ids = ds.tile_ids[:limit]
    loader = DataLoader(ds, batch_size=8, shuffle=False, num_workers=NW)
    ids = []
    t0 = _time.time()
    n_batches = 0
    for b in loader:
        loc_p, dmg_p = tta_probs(model, b["pre"].to(DEVICE), b["post"].to(DEVICE))
        loc_np = loc_p.cpu().numpy().astype(np.float16)
        dmg_np = dmg_p.cpu().numpy().astype(np.float16)
        for i in range(loc_np.shape[0]):
            tid = b["tile_id"][i]
            np.savez_compressed(Path("/tmp") / f"eval_{split}_{tid}.npz",
                                loc=loc_np[i], dmg=dmg_np[i])
            ids.append(tid)
        n_batches += 1
        if n_batches % 5 == 0:
            el = _time.time() - t0
            mark(f"cache:{tag}", extra=f"{len(ids)} tiles, {el/60:.1f}min, "
                                        f"~{el/max(len(ids),1):.1f}s/tile")
    mark(f"cache:{tag}-done", extra=f"{len(ids)} tiles in {(_time.time()-t0)/60:.1f}min")
    return ids


def cached_ids(tag, split):
    prefix = f"{tag}_{split}_"
    return sorted(p.name.removeprefix(prefix).removesuffix(".npz")
                  for p in Path("/tmp").glob(prefix + "*.npz"))


mark("modelkit:done")

In [ ]:
import gc, logging, math, shutil, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, WeightedRandomSampler
from rescuevision.utils.seed import set_seed

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
name = "joint_damage_serox50_s1"
ENCODER = "se_resnext50_32x4d"
ENCWEIGHTS = "imagenet"
SIZE = 608
SEED = 42
EPOCHS = 16          # adaptive: may shrink after epoch 1 to fit budget
MIN_EPOCHS = 12
PATIENCE = 6
BS = 8
TIME_BUDGET_H = 9.5
LR = 0.0002
RESUME_CKPT = /kaggle/input/m1-resume-ep9/last.pt   # None or path (mount a private dataset with last.pt)
run_dir = Path("/tmp/run") / name
(run_dir / "checkpoints").mkdir(parents=True, exist_ok=True)
(run_dir / "logs").mkdir(parents=True, exist_ok=True)
ckpt_out = WORK / name / "checkpoints"   # persistent: survives kernel death
ckpt_out.mkdir(parents=True, exist_ok=True)


def _rss_gb():
    try:
        with open("/proc/self/status") as f:
            for l in f:
                if l.startswith("VmRSS"):
                    return int(l.split()[1]) / 1e6
    except Exception:
        return -1.0
    return -1.0


import torch as _torch
import torch.nn as _nn
import torch.nn.functional as _F
import segmentation_models_pytorch as _smp


class SiameseUNet(_nn.Module):
    """Shared-encoder siamese U-Net with fused pre/post features (fixed decoder)."""

    def __init__(self, encoder_name="resnet34", encoder_weights="imagenet",
                 fusion_mode="concat_absdiff", classes=1):
        super().__init__()
        self.fusion_mode = fusion_mode
        self.classes = classes

        base_unet = _smp.Unet(encoder_name=encoder_name, encoder_weights=encoder_weights,
                              in_channels=3, classes=classes)
        self.encoder = base_unet.encoder
        encoder_channels = self._get_encoder_channels()
        bottleneck_ch = encoder_channels[-1]

        if fusion_mode == "concat":
            fusion_mult = 2
        elif fusion_mode == "absdiff":
            fusion_mult = 1
        else:
            fusion_mult = 3   # concat_absdiff

        skip_channels = list(reversed(encoder_channels[:-1]))
        decoder_channels = [256, 128, 64, 32]

        self.skip_convs = _nn.ModuleList()
        for enc_ch in skip_channels:
            self.skip_convs.append(_nn.Sequential(
                _nn.Conv2d(enc_ch * fusion_mult, enc_ch, 1, bias=False),
                _nn.BatchNorm2d(enc_ch),
                _nn.ReLU(inplace=True),
            ))

        self.decoder = _nn.ModuleList()
        in_ch = bottleneck_ch * fusion_mult + skip_channels[0]
        for i, dec_ch in enumerate(decoder_channels):
            self.decoder.append(_nn.Sequential(
                _nn.Conv2d(in_ch, dec_ch, 3, padding=1, bias=False),
                _nn.BatchNorm2d(dec_ch),
                _nn.ReLU(inplace=True),
                _nn.Conv2d(dec_ch, dec_ch, 3, padding=1, bias=False),
                _nn.BatchNorm2d(dec_ch),
                _nn.ReLU(inplace=True),
            ))
            if i + 1 < len(decoder_channels):
                in_ch = dec_ch + skip_channels[i + 1]

        self.segmentation_head = _nn.Conv2d(decoder_channels[-1], classes, 1)

    def _get_encoder_channels(self):
        channels = []
        dummy = _torch.randn(1, 3, 64, 64)
        with _torch.no_grad():
            features = self.encoder(dummy)
        return [f.shape[1] for f in features]

    def _fuse(self, a, b):
        if self.fusion_mode == "concat":
            return _torch.cat([a, b], dim=1)
        if self.fusion_mode == "absdiff":
            return _torch.abs(b - a)
        return _torch.cat([a, b, _torch.abs(b - a)], dim=1)

    def forward(self, pre_image, post_image):
        features_pre = self.encoder(pre_image)
        features_post = self.encoder(post_image)
        x = self._fuse(features_pre[-1], features_post[-1])
        for i, (dec_block, skip_conv) in enumerate(zip(self.decoder, self.skip_convs)):
            x = _F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
            skip = skip_conv(self._fuse(features_pre[-(i + 2)], features_post[-(i + 2)]))
            if x.shape[2:] != skip.shape[2:]:
                x = _F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=False)
            x = dec_block(_torch.cat([x, skip], dim=1))
        if x.shape[2:] != pre_image.shape[2:]:
            x = _F.interpolate(x, size=pre_image.shape[2:], mode="bilinear", align_corners=False)
        return self.segmentation_head(x)


print("SiameseUNet (fixed decoder) defined")

class SEGate(_torch.nn.Module):
    """Squeeze-excitation channel gate for a skip pathway."""

    def __init__(self, ch, reduction=8):
        super().__init__()
        self.fc = _torch.nn.Sequential(
            _torch.nn.AdaptiveAvgPool2d(1),
            _torch.nn.Flatten(),
            _torch.nn.Linear(ch, max(ch // reduction, 8)), _torch.nn.ReLU(inplace=True),
            _torch.nn.Linear(max(ch // reduction, 8), ch), _torch.nn.Sigmoid())

    def forward(self, x):
        return x * self.fc(x).view(x.shape[0], x.shape[1], 1, 1)


class SiameseUNetSE(SiameseUNet):
    """SiameseUNet + SE gates on fused skips. New params only in skip_gates,
    so plain-SiameseUNet checkpoints remain loadable with the base class."""

    def __init__(self, *a, **k):
        super().__init__(*a, **k)
        enc_chs = self._get_encoder_channels()
        skip_channels = list(reversed(enc_chs[:-1]))
        self.skip_gates = _torch.nn.ModuleList([SEGate(ch) for ch in skip_channels])

    def forward(self, pre_image, post_image):
        features_pre = self.encoder(pre_image)
        features_post = self.encoder(post_image)
        x = self._fuse(features_pre[-1], features_post[-1])
        for i, (dec_block, skip_conv, gate) in enumerate(
                zip(self.decoder, self.skip_convs, self.skip_gates)):
            x = _F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
            skip = gate(skip_conv(self._fuse(features_pre[-(i + 2)], features_post[-(i + 2)])))
            if x.shape[2:] != skip.shape[2:]:
                x = _F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=False)
            x = dec_block(_torch.cat([x, skip], dim=1))
        if x.shape[2:] != pre_image.shape[2:]:
            x = _F.interpolate(x, size=pre_image.shape[2:], mode="bilinear", align_corners=False)
        return self.segmentation_head(x)


print("SiameseUNetSE (skip SE gates) defined")

# ---- damage-biased tile sampler: grade>=2 tiles 3x, grade-1-only 1.5x ----
def damage_biased_weights(tile_ids, masks_dir):
    from PIL import Image as PILImage
    w = np.ones(len(tile_ids), dtype=np.float64)
    for i, tid in enumerate(tile_ids):
        d = np.array(PILImage.open(masks_dir / f"{tid}.png"))
        if (d >= 2).any():
            w[i] = 3.0
        elif (d == 1).any():
            w[i] = 1.5
    return w


class EMA:
    """Exponential moving average of a model state_dict (floats averaged,
    integer buffers copied). No framework dependency."""

    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone().float() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach().float(), alpha=1 - self.decay)
            else:
                self.shadow[k].copy_(v.detach())

    def state_dict_into(self, model):
        base = model.state_dict()
        return {k: self.shadow[k].to(base[k].dtype) for k in base}


class SWASoup:
    """Equal-weight average of collected state_dicts (last 25% of epochs)."""

    def __init__(self):
        self.acc = None
        self.n = 0

    def add(self, sd):
        if self.acc is None:
            self.acc = {k: v.detach().clone().float() for k, v in sd.items()}
        else:
            for k, v in sd.items():
                self.acc[k] += v.detach().float()
        self.n += 1

    def state_dict_into(self, model):
        assert self.n > 0, "swa soup empty"
        base = model.state_dict()
        return {k: (self.acc[k] / self.n).to(base[k].dtype) for k in base}


set_seed(SEED)
split_csv = PROCESSED / "metadata" / "splits_event_aware.csv"
train_ds = JointDamageDataset(PROCESSED, split_csv, "train", joint_pair_transforms("train", SIZE))
val_ds = JointDamageDataset(PROCESSED, split_csv, "val", joint_pair_transforms("val", SIZE))

sampler = None
if True:
    w = damage_biased_weights(train_ds.tile_ids, PROCESSED / "masks/damage_multiclass_dil")
    print(f"sampler weights: 3x={int((w == 3).sum())} 1.5x={int((w == 1.5).sum())} 1x={int((w == 1).sum())}")
    sampler = WeightedRandomSampler(torch.from_numpy(w), num_samples=len(train_ds), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=BS, sampler=sampler,
                              num_workers=NW, pin_memory=False, persistent_workers=bool(NW))
else:
    train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True,
                              num_workers=NW, pin_memory=False, persistent_workers=bool(NW))
val_loader = DataLoader(val_ds, batch_size=BS, shuffle=False, num_workers=NW, pin_memory=False)
print(f"train={len(train_ds)} val={len(val_ds)}")
mark("joint:data", extra=f"train={len(train_ds)} val={len(val_ds)} sampler={sampler is not None}")

# --- winner-style 1/freq damage weights + loc pos_weight (from mask sample) ---
dmg_counts = np.zeros(4, dtype=np.int64); loc_pos = 0; loc_tot = 0
for tid in train_ds.tile_ids[:1200]:
    d = _load_arr(PROCESSED / "masks/damage_multiclass_dil" / f"{tid}.png")
    l = _load_arr(PROCESSED / "masks/building_binary" / f"{tid}.png")
    v = d[(d >= 1) & (d <= 4)]
    if v.size:
        dmg_counts += np.bincount(v.flatten().astype(np.int64), minlength=5)[1:5]
    loc_pos += int((l > 0).sum()); loc_tot += l.size
assert dmg_counts.sum() > 0
freq = dmg_counts / dmg_counts.sum()
print("empirical class freq (reference):", [round(x, 4) for x in freq.tolist()])
# research-validated fixed recipe (Improved-MambaBDA): alpha=[0.6,1.6,1.1,1.1], gamma=1.5
DMG_W = torch.tensor([0.6, 1.6, 1.1, 1.1], dtype=torch.float32, device=DEVICE)
loc_frac = loc_pos / max(loc_tot, 1)
LOC_PW = torch.tensor(min(15.0, max(1.0, (1 - loc_frac) / max(loc_frac, 1e-4))), device=DEVICE)
print("dmg weights (fixed recipe):", DMG_W.tolist(), "| loc pw:", round(float(LOC_PW), 2))
mark("joint:weights", extra=f"w={DMG_W.tolist()} pw={float(LOC_PW):.1f}")

model = SiameseUNetSE(encoder_name=ENCODER, encoder_weights=ENCWEIGHTS,
                      fusion_mode="concat_absdiff", classes=5).to(DEVICE)
N_GPU = torch.cuda.device_count() if DEVICE.type == "cuda" else 0
raw_model = model
if N_GPU > 1:
    model = nn.DataParallel(model)
    print(f"DataParallel over {N_GPU} GPUs")
mark("joint:model", extra=f"encoder={ENCODER} w={ENCWEIGHTS} gpu={N_GPU}")


def _sd(m):
    return m.module.state_dict() if isinstance(m, nn.DataParallel) else m.state_dict()


def _load_sd(m, sd):
    (m.module if isinstance(m, nn.DataParallel) else m).load_state_dict(sd)


def _lovasz_grad(gt_sorted):
    """Gradient of the Jaccard index wrt sorted errors (Berman et al. 2018)."""
    p = len(gt_sorted)
    gts = gt_sorted.sum()
    intersection = gts - gt_sorted.float().cumsum(0)
    union = gts + (1 - gt_sorted).float().cumsum(0)
    jaccard = 1.0 - intersection / torch.clamp(union, min=1e-12)
    if p > 1:
        jaccard[1:p] = jaccard[1:p] - jaccard[0:-1]
    return torch.clamp(jaccard, min=0.0)


def _lovasz_softmax(logits, labels, ignore_index=255):
    """Batch-flat Lovasz-softmax over present classes; logits (N,C,H,W) fp32."""
    valid = labels != ignore_index
    if valid.sum() == 0:
        return logits.sum() * 0.0
    probs = torch.softmax(logits, dim=1)
    losses = []
    for c in range(logits.shape[1]):
        fg = ((labels == c) & valid).reshape(-1).float()
        if fg.sum() == 0:
            continue
        errors = (fg - probs[:, c].reshape(-1)).abs()
        errors_sorted, perm = torch.sort(errors, descending=True)
        fg_sorted = fg[perm]
        losses.append(torch.dot(errors_sorted, _lovasz_grad(fg_sorted)))
    if not losses:
        return logits.sum() * 0.0
    return torch.stack(losses).mean()


class MaskedFocalDiceLoss(nn.Module):
    """Focal CE (alpha per class, gamma) + soft Dice + Lovasz-softmax over the
    4 damage grades, graded pixels only. Recipe: alpha=[0.6,1.6,1.1,1.1]
    gamma=1.5 (Improved-MambaBDA), Dice weight 1.0, Lovasz weight 0.5."""

    def __init__(self, weight, gamma=1.5, dice_weight=1.0, lovasz_weight=0.5, ignore_index=255):
        super().__init__()
        self.register_buffer("weight", weight)
        self.gamma = gamma; self.dice_weight = dice_weight
        self.lovasz_weight = lovasz_weight; self.ignore_index = ignore_index

    def forward(self, logits, target):
        valid = target != self.ignore_index
        if valid.sum() == 0:
            return logits.sum() * 0.0
        tgt = torch.where(valid, target, torch.zeros_like(target))
        logp = F.log_softmax(logits, dim=1)
        ce = F.nll_loss(logp, tgt, weight=self.weight, reduction="none")
        p_t = logp.gather(1, tgt.unsqueeze(1)).squeeze(1).exp()
        focal = (((1 - p_t) ** self.gamma) * ce * valid).sum() / valid.sum().clamp(min=1)
        probs = logp.exp()
        dice = 0.0
        for c in range(logits.shape[1]):
            p = probs[:, c][valid]; g = (tgt[valid] == c).float()
            dice = dice + (1 - (2 * (p * g).sum() + 1) / (p.sum() + g.sum() + 1))
        dice = dice / logits.shape[1]
        lov = _lovasz_softmax(logits.float(), target, self.ignore_index)
        return focal + self.dice_weight * dice + self.lovasz_weight * lov


dmg_loss_fn = MaskedFocalDiceLoss(DMG_W, gamma=1.5)


def joint_loss(out, loc_t, dmg_t):
    loc_logits, dmg_logits = out[:, 0], out[:, 1:]
    bce = F.binary_cross_entropy_with_logits(loc_logits, loc_t, pos_weight=LOC_PW)
    prob = torch.sigmoid(loc_logits)
    inter = (prob * loc_t).sum(dim=(1, 2)); card = prob.sum(dim=(1, 2)) + loc_t.sum(dim=(1, 2))
    dice = (1 - (2 * inter + 1) / (card + 1)).mean()
    loc_l = 0.5 * bce + 0.5 * dice
    dmg_l = dmg_loss_fn(dmg_logits, dmg_t)
    return loc_l + dmg_l, float(loc_l.detach()), float(dmg_l.detach())


# ---- probe: 2 train steps before committing to the loop (OOM -> halve BS once) ----
def _two_steps():
    _it = iter(train_loader)
    for _ in range(2):
        _b = next(_it)
        with torch.amp.autocast(DEVICE.type, enabled=(DEVICE.type == "cuda")):
            _o = model(_b["pre"].to(DEVICE), _b["post"].to(DEVICE))
            _l, _, _ = joint_loss(_o, _b["loc"].to(DEVICE), _b["dmg"].to(DEVICE))
        _l.backward()
        model.zero_grad(set_to_none=True)

try:
    _two_steps()
except RuntimeError as _e:
    if "out of memory" in str(_e).lower() and DEVICE.type == "cuda":
        torch.cuda.empty_cache()
        BS = max(1, BS // 2)
        mark("probe:oom-halve-bs", extra=f"BS->{BS}")
        if sampler is not None:
            train_loader = DataLoader(train_ds, batch_size=BS, sampler=sampler,
                                      num_workers=NW, pin_memory=False,
                                      persistent_workers=bool(NW))
        else:
            train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True,
                                      num_workers=NW, pin_memory=False,
                                      persistent_workers=bool(NW))
        _two_steps()
    else:
        raise
try:
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
        mark("probe:ok", extra=f"peak={torch.cuda.max_memory_allocated()/1e9:.1f}GB")
    else:
        mark("probe:ok", extra="cpu")
except Exception:
    mark("probe:FAIL", extra=traceback.format_exc()[-1200:])
    raise


@torch.no_grad()
def joint_val(model, loader):
    model.eval()
    inter = union = 0
    tp = np.zeros(4); fp = np.zeros(4); fn = np.zeros(4)
    for b in loader:
        pre = b["pre"].to(DEVICE); post = b["post"].to(DEVICE)
        with torch.amp.autocast(DEVICE.type, enabled=(DEVICE.type == "cuda")):
            out = model(pre, post)
        loc_p = torch.sigmoid(out[:, 0].float()) > 0.5
        loc_t = b["loc"].to(DEVICE) > 0.5
        inter += (loc_p & loc_t).sum().item(); union += (loc_p | loc_t).sum().item()
        dmg_t = b["dmg"].to(DEVICE); valid = dmg_t != 255
        if valid.sum():
            pred = out[:, 1:].float().argmax(1)
            for c in range(4):
                p = (pred == c) & valid; g = (dmg_t == c) & valid
                tp[c] += (p & g).sum().item(); fp[c] += (p & ~g).sum().item(); fn[c] += (g & ~p).sum().item()
    model.train()
    loc_iou = inter / max(union, 1)
    f1s = [2 * tp[c] / max(2 * tp[c] + fp[c] + fn[c], 1) for c in range(4)]
    return {"loc_iou": loc_iou, "dmg_macro_f1": float(np.mean(f1s)),
            "class_f1": [round(f, 4) for f in f1s]}


opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
steps_per_epoch = len(train_loader)


def make_lr_lambda(total_steps, warm):
    def fn(step):
        if step < warm:
            return (step + 1) / warm
        prog = (step - warm) / max(total_steps - warm, 1)
        return 0.5 * (1 + math.cos(math.pi * min(prog, 1.0)))
    return fn


sched = torch.optim.lr_scheduler.LambdaLR(opt, make_lr_lambda(EPOCHS * steps_per_epoch, steps_per_epoch))
scaler = torch.amp.GradScaler(DEVICE.type, enabled=(DEVICE.type == "cuda"))
ema = EMA(raw_model)
soup = SWASoup()

from rescuevision.training.logging_utils import CSVLogger
csv = CSVLogger(run_dir / "logs")
T0 = time.time()
best = -1.0; best_epoch = 0; bad = 0
start_epoch = 0

# ---- optional resume from a persistent last.pt (mounted private dataset) ----
if RESUME_CKPT is not None:
    _ck = torch.load(RESUME_CKPT, map_location=DEVICE, weights_only=False)
    _load_sd(model, _ck["model_state_dict"])
    opt.load_state_dict(_ck["optimizer_state_dict"])
    start_epoch = int(_ck.get("epoch", 0))
    best = float(_ck.get("best", -1.0)); best_epoch = int(_ck.get("best_epoch", 0))
    bad = int(_ck.get("bad", 0))
    if _ck.get("ema_shadow"):
        for k in ema.shadow:
            ema.shadow[k] = _ck["ema_shadow"][k].to(ema.shadow[k].device).clone()
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, make_lr_lambda(EPOCHS * steps_per_epoch, steps_per_epoch),
        last_epoch=start_epoch * steps_per_epoch - 1)
    mark("joint:resumed", extra=f"from ep{start_epoch} best={best:.4f} bad={bad} "
                                f"src={RESUME_CKPT}")
    print(f"RESUMED from epoch {start_epoch}, best={best:.4f}")
mark("joint:train-start", extra=f"train={len(train_ds)} epochs={EPOCHS} enc={ENCODER} "
                                f"resume={start_epoch > 0}")
epoch = start_epoch
while epoch < EPOCHS:
    epoch += 1
    model.train()
    tr_loss = 0.0; nb = 0
    ep_t0 = time.time()
    for b in train_loader:
        pre = b["pre"].to(DEVICE); post = b["post"].to(DEVICE)
        loc_t = b["loc"].to(DEVICE); dmg_t = b["dmg"].to(DEVICE)
        with torch.amp.autocast(DEVICE.type, enabled=(DEVICE.type == "cuda")):
            out = model(pre, post)
            loss, l_loc, l_dmg = joint_loss(out, loc_t, dmg_t)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        sched.step()
        ema.update(raw_model)
        tr_loss += loss.item(); nb += 1
    vm = joint_val(model, val_loader)
    ep_secs = time.time() - ep_t0

    combined = 0.5 * vm["loc_iou"] + 0.5 * vm["dmg_macro_f1"]
    row = {"train_loss": tr_loss / max(nb, 1), "val_loss": 0.0,
           "loc_iou": vm["loc_iou"], "dmg_macro_f1": vm["dmg_macro_f1"],
           "combined": combined, "lr": opt.param_groups[0]["lr"],
           "epoch_time": ep_secs}
    csv.log(row, epoch)
    mark(f"joint:ep{epoch}", extra=str({"combined": round(combined, 4), "rss_gb": round(_rss_gb(), 1),
                                        **{k: v for k, v in row.items() if k != "combined"}})[:400])
    # durable snapshot EVERY epoch: /tmp copy + persistent /kaggle/working copy
    _snap = {"model_state_dict": _sd(model), "epoch": epoch,
             "optimizer_state_dict": opt.state_dict(),
             "combined": combined, "val": vm,
             "best": best, "best_epoch": best_epoch, "bad": bad,
             "ema_shadow": {k: v.detach().clone() for k, v in ema.shadow.items()},
             "encoder": ENCODER}
    torch.save(_snap, run_dir / "checkpoints" / "last.pt")
    torch.save(_snap, ckpt_out / "last.pt")
    if combined > best:
        best = combined; best_epoch = epoch; bad = 0
        _snap["best"] = best; _snap["best_epoch"] = best_epoch
        torch.save({"model_state_dict": _sd(model), "epoch": epoch,
                    "combined": combined, "val": vm,
                    "dmg_weights": DMG_W.cpu(), "loc_pos_weight": float(LOC_PW),
                    "encoder": ENCODER, "arch": "SiameseUNetSE"},
                   run_dir / "checkpoints" / "best.pt")
        shutil.copyfile(run_dir / "checkpoints" / "best.pt", ckpt_out / "best.pt")
        print(f"epoch {epoch} NEW BEST combined={combined:.4f}")
    else:
        bad += 1
    del _snap
    gc.collect()
    # release freed heap back to the OS: ~33k image decodes/epoch fragment the
    # C allocator; without this RSS grew ~2.9GB/epoch until the host OOM-kill
    try:
        import ctypes
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    _rss = _rss_gb()
    if _rss > 26.0:
        mark("joint:rss-guard-stop",
             extra=f"ep{epoch} rss={_rss:.1f}GB > 26GB - clean stop at epoch "
                   f"boundary; resume with RESUME_CKPT to continue")
        break

    if epoch >= max(MIN_EPOCHS, math.ceil(EPOCHS * 0.75)):
        soup.add(_sd(model))

    # adaptive cosine: after epoch 1, shrink the schedule to what the budget allows
    if epoch == 1 and start_epoch == 0:
        elapsed = time.time() - T0
        feasible = int((TIME_BUDGET_H * 3600 - elapsed) / max(ep_secs, 1))
        feasible = max(MIN_EPOCHS, min(EPOCHS, feasible))
        if feasible < EPOCHS:
            mark("joint:adaptive-epochs", extra=f"{EPOCHS}->{feasible} ({ep_secs:.0f}s/ep)")
            EPOCHS = feasible
            sched = torch.optim.lr_scheduler.LambdaLR(
                opt, make_lr_lambda(EPOCHS * steps_per_epoch, steps_per_epoch),
                last_epoch=steps_per_epoch - 1)

    el = (time.time() - T0) / 3600
    if bad >= PATIENCE:
        mark("joint:early-stop", extra=f"ep{epoch} best_ep{best_epoch} {el:.1f}h")
        break
    if el > TIME_BUDGET_H:
        mark("joint:time-guard", extra=f"ep{epoch} {el:.1f}h > {TIME_BUDGET_H}h")
        break
mark("joint:train-done", extra=f"best={best:.4f} @ep{best_epoch}")

# ---- pick final weights: best_raw vs EMA vs SWA soup, judged on val ----
candidates = {"best_raw": torch.load(run_dir / "checkpoints" / "best.pt",
                                     map_location=DEVICE, weights_only=False)}
ema_sd = ema.state_dict_into(raw_model)
try:
    _load_sd(model, ema_sd)
    vm_ema = joint_val(model, val_loader)
    candidates["ema"] = {"model_state_dict": ema_sd, "epoch": epoch, "val": vm_ema}
except Exception:
    mark("ema:eval-fail", extra=traceback.format_exc()[-600:])
if soup.n > 0:
    try:
        _load_sd(model, soup.state_dict_into(raw_model))
        vm_swa = joint_val(model, val_loader)
        candidates["swa"] = {"model_state_dict": soup.state_dict_into(raw_model),
                             "epoch": epoch, "val": vm_swa}
    except Exception:
        mark("swa:eval-fail", extra=traceback.format_exc()[-600:])

for cname, cand in candidates.items():
    print(cname, "->", cand.get("val"))
final_name = max(candidates, key=lambda k: 0.5 * candidates[k]["val"]["loc_iou"]
                 + 0.5 * candidates[k]["val"]["dmg_macro_f1"])
final = candidates[final_name]
final["final_choice"] = final_name
final["candidate_vals"] = {k: v.get("val") for k, v in candidates.items()}
final["dmg_weights"] = DMG_W.cpu()
final["loc_pos_weight"] = float(LOC_PW)
final["encoder"] = ENCODER
torch.save(final, run_dir / "checkpoints" / "final.pt")
mark("joint:final", extra=f"{final_name} val={final['val']}")
print("FINAL:", final_name, final["val"])

In [ ]:
# ---- copy results out + cache probs for official eval ----
import shutil
shutil.copytree(f"/tmp/run/joint_damage_serox50_s1", WORK / "joint_damage_serox50_s1", dirs_exist_ok=True)
mark("copyout:joint_damage_serox50_s1")

model = build_model("se_resnext50_32x4d", "none", se_gates=True)
ckpt = torch.load(WORK / "joint_damage_serox50_s1" / "checkpoints" / "final.pt",
                  map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
print("final ckpt:", ckpt.get("final_choice"), "val", ckpt.get("val"))

split_csv = PROCESSED / "metadata" / "splits_event_aware.csv"
val_ds = JointDamageDataset(PROCESSED, split_csv, "val", joint_pair_transforms("val", SIZE))
ids_val = cache_probs(model, val_ds, "val", limit=250, tag="eval")
test_ds = JointDamageDataset(PROCESSED, split_csv, "test", joint_pair_transforms("val", SIZE))
ids_test = cache_probs(model, test_ds, "test", limit=None, tag="eval")
del model
mark("eval:probs-cached", extra=f"val={len(ids_val)} test={len(ids_test)}")

In [ ]:
# ---- official scoring over cached probs: threshold sweep + winner rule +
# ---- inference-time minor dilation, against original AND dilated targets.
import json
from functools import lru_cache
import numpy as np
from PIL import Image
from scipy.ndimage import binary_dilation
import cv2

SCORING_SPLIT = "val"
THR_LIST = [round(t, 2) for t in np.arange(0.3, 0.85 + 1e-9, 0.05)]
DILATE_ITERS = 2
TAG = "eval"


@lru_cache(maxsize=None)
def load_gt(tid):
    bld = np.array(Image.open(PROCESSED / "masks/building_binary" / f"{tid}.png"))
    dmg = np.array(Image.open(PROCESSED / "masks/damage_multiclass" / f"{tid}.png"))
    dmgd = np.array(Image.open(PROCESSED / "masks/damage_multiclass_dil" / f"{tid}.png"))
    return bld, dmg, dmgd


def to_native_2d(arr, shape):
    if arr.shape == shape:
        return arr
    return cv2.resize(arr.astype(np.float32), (shape[1], shape[0]),
                      interpolation=cv2.INTER_LINEAR)


def to_native_3d(arr, shape):
    """(C,h,w) channels-first -> bilinear resample each channel to shape."""
    if arr.shape[1:] == shape:
        return arr
    return np.stack([cv2.resize(arr[i].astype(np.float32), (shape[1], shape[0]),
                                interpolation=cv2.INTER_LINEAR)
                     for i in range(arr.shape[0])])


@lru_cache(maxsize=None)
def native_preds(tid, prefix=None):
    """Per-tile native-resolution inputs, computed once and shared by all
    scoring variants: (bld, dmg, dmgd, loc_prob, grade_1to4).
    prefix defaults to the current TAG_SPLIT (pass e.g. 'eval_test' to read
    another split's cached probs)."""
    prefix = prefix or f"{TAG}_{SCORING_SPLIT}"
    z = np.load(Path("/tmp") / f"{prefix}_{tid}.npz")
    bld, dmg, dmgd = load_gt(tid)
    shape = bld.shape
    loc = to_native_2d(z["loc"].astype(np.float32), shape)
    dmg_p = to_native_3d(z["dmg"].astype(np.float32), shape)  # (4,H,W)
    grade = np.argmax(dmg_p, axis=0) + 1
    return bld, dmg, dmgd, loc, grade


def winner_building_mask(loc_prob, grade):
    """xView2-winner conditional rule, EXACT (create_submission.py): building if
    loc>0.38 OR (loc>0.13 & grade in {2 minor, 3 major}) OR (loc>0.14 & grade>=2)."""
    return ((loc_prob > 0.38) |
            ((loc_prob > 0.13) & (grade >= 2) & (grade <= 3)) |
            ((loc_prob > 0.14) & (grade >= 2)))


def dilate_minor_pred(grade, iters=DILATE_ITERS):
    """winner-exact: 5x5 dilation of the MINOR(2) mask; only no-damage(1)
    pixels inside the dilated mask flip to minor (bg and 3/4 untouched)."""
    if iters <= 0:
        return grade, np.zeros_like(grade, dtype=bool)
    st = np.ones((5, 5), dtype=bool)
    grow = binary_dilation(grade == 2, structure=st) & (grade == 1)
    out = grade.copy()
    out[grow] = 2
    return out, grow


def score_variant(ids, thr, rule, dilate, target_key):
    """rule: 'plain' (thr on loc) or 'winner'. target_key: 'dmg' (original
    annotations = official) or 'dmgd' (dilated training labels, reference)."""
    sc = XView2PixelScorer()
    for tid in ids:
        bld, dmg, dmgd, loc, grade = native_preds(tid)
        build = winner_building_mask(loc, grade) if rule == "winner" else (loc > thr)
        grade = np.where(build, grade, 0)
        if dilate:
            grade, grow = dilate_minor_pred(grade)
            build = build | grow
        tgt = dmg if target_key == "dmg" else dmgd
        sc.add(build.astype(np.uint8), grade.astype(np.uint8),
               bld.astype(np.uint8), tgt.astype(np.uint8))
    return sc.summary()


ids = cached_ids(TAG, SCORING_SPLIT)
assert len(ids) > 0, "no cached probs found"
print(f"scoring {len(ids)} tiles of split '{SCORING_SPLIT}'")
_ = [native_preds(t) for t in ids]  # warm the per-tile cache once

results = {}
for target_key in ("dmg", "dmgd"):
    variants = [("plain", t) for t in THR_LIST] + [("winner", None)]
    for rule, thr in variants:
        for dilate in ((False, True) if DILATE_ITERS > 0 else (False,)):
            key = f"tgt={target_key}|rule={rule}|thr={thr}|dilate={int(dilate)}"
            r = score_variant(ids, thr, rule, dilate, target_key)
            results[key] = r
            print(f"{key:44s} score={r['score']:.4f} loc={r['localization_f1']:.4f} "
                  f"dmg={r['damage_f1']:.4f}")

best_key = max(results, key=lambda k: results[k]["score"])
official = results[best_key]
print("\nBEST RECIPE:", best_key)
print(json.dumps(official, indent=2))

OFFICIAL_RESULTS = {"best_recipe": best_key, "best": official,
                    "n_tiles": len(ids), "split": SCORING_SPLIT,
                    "all_variants": results}
(Path("/tmp") / f"{TAG}_official_{SCORING_SPLIT}.json").write_text(
    json.dumps(OFFICIAL_RESULTS, indent=1))
mark(f"official:{SCORING_SPLIT}-done",
     extra=f"best={best_key} score={official['score']:.4f}")

In [ ]:
# ---- threshold selection on VAL (official score), then official TEST score ----
val_official = json.loads(Path("/tmp/eval_official_val.json").read_text())
best_recipe = val_official["best_recipe"]
print("val-selected recipe:", best_recipe)
rule = best_recipe.split("|")[1].split("=")[1]
thr_s = best_recipe.split("|")[2].split("=")[1]
thr = None if thr_s == "None" else float(thr_s)
dilate = best_recipe.split("|")[3].split("=")[1] == "1"

ids = cached_ids("eval", "test")
assert len(ids) > 0, "no test probs cached"
sc = XView2PixelScorer()
pix_iou_inter = pix_iou_union = 0
f1tp = np.zeros(4); f1fp = np.zeros(4); f1fn = np.zeros(4)
for tid in ids:
    bld, dmg, dmgd, loc, grade = native_preds(tid, "eval_test")
    build = winner_building_mask(loc, grade) if rule == "winner" else (loc > thr)
    grade = np.where(build, grade, 0)
    if dilate:
        grade, grow = dilate_minor_pred(grade)
        build = build | grow
    sc.add(build.astype(np.uint8), grade.astype(np.uint8),
           bld.astype(np.uint8), dmg.astype(np.uint8))
    pix_iou_inter += int(np.logical_and(build, bld > 0).sum())
    pix_iou_union += int(np.logical_or(build, bld > 0).sum())
    inside = dmg > 0
    dp_in = np.where(build, grade, 0)[inside]
    dt_in = dmg[inside]
    for c in range(4):
        p = dp_in == c + 1; g = dt_in == c + 1
        f1tp[c] += int((p & g).sum()); f1fp[c] += int((p & ~g).sum()); f1fn[c] += int((g & ~p).sum())

official_test = sc.summary()
pix_f1s = [2 * f1tp[c] / max(2 * f1tp[c] + f1fp[c] + f1fn[c], 1) for c in range(4)]
summary = {
    "run": "joint_damage_serox50_s1", "encoder": ckpt.get("encoder"),
    "final_choice": ckpt.get("final_choice"), "best_epoch": int(ckpt.get("epoch", -1)),
    "candidate_vals": ckpt.get("candidate_vals"),
    "val_selected_recipe": best_recipe,
    "test": {"official": official_test,
             "pixel_loc_iou": round(pix_iou_inter / max(pix_iou_union, 1), 4),
             "pixel_dmg_f1": [round(f, 4) for f in pix_f1s],
             "pixel_dmg_macro_f1": round(float(np.mean(pix_f1s)), 4)},
    "n_test_tiles": len(ids),
}
(WORK / "joint_damage_serox50_s1" / "official_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
mark("joint:official-eval-done",
     extra=f"score={official_test['score']:.4f} loc={official_test['localization_f1']:.4f} "
           f"dmg={official_test['damage_f1']:.4f}")

In [ ]:
import shutil as _shutil
_shutil.copytree(f"/tmp/run/joint_damage_serox50_s1", WORK / "joint_damage_serox50_s1", dirs_exist_ok=True)
print("results copied to", WORK / "joint_damage_serox50_s1")
mark("copyout:joint_damage_serox50_s1")
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

log_df = pd.read_csv(next((WORK / "joint_damage_serox50_s1" / "logs").glob("*.csv")))
metric = "combined"
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(log_df["epoch"], log_df["train_loss"], label="train")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)
if metric in log_df.columns:
    axes[1].plot(log_df["epoch"], log_df[metric], color="tab:green")
    axes[1].set_title(f"val {metric} (best={log_df[metric].max():.4f} @ ep {int(log_df.loc[log_df[metric].idxmax(), 'epoch'])})")
axes[1].set_xlabel("epoch"); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(WORK / "joint_damage_serox50_s1" / "training_curves.png", dpi=120, bbox_inches="tight")
plt.show()